# Quadrotor (aka "the drone")

## EOM derivation

Do all imports.

In [ ]:
import sympy as sym
import numpy as np
from sympy import *
from scipy.linalg import solve_continuous_are          
import matplotlib.pyplot as plt          

np.set_printoptions(suppress=True) # Suppress the use of scientific notation when printing small numbers

## Dynamic model

Define physical parameters.

In [1]:
import numpy as np

# Compute inertia tensor for a quadcopter with:
# - 735 g central point mass at origin
# - 60.7 g motors at tips of 4 arms
# - 4 rods (each 65 mm long, 161.8 g) forming an X/Y cross

m_center = 735.0/1000     # kg
m_motor  = 60.7/1000      # kg
m_rod    = 161.8/1000     # kg
L        = 65.0/1000      # m

# Motors: point masses at ±L on x and y axes
motor_positions = [
    np.array([ L, 0, 0]),
    np.array([-L, 0, 0]),
    np.array([ 0, L, 0]),
    np.array([ 0,-L, 0])
]

I = np.zeros((3,3))   # central point mass contributes zero

# Add motor inertias (point mass formula: I = m (r^2 I - r r^T))
for r in motor_positions:
    r2 = np.dot(r,r)
    I += m_motor * (r2*np.eye(3) - np.outer(r,r))

# Rod inertia: rod from origin to (L,0,0) has Iy = Iz = (1/3)mL²
I_rod = (1/3)*m_rod*L**2

# Two rods along ±x → contribute to Iy, Iz
I[1,1] += 2*I_rod
I[2,2] += 2*I_rod

# Two rods along ±y → contribute to Ix, Iz
I[0,0] += 2*I_rod
I[2,2] += 2*I_rod

# Output principal inertias
Jx, Jy, Jz = I[0,0], I[1,1], I[2,2]

I, Jx, Jy, Jz

(array([[0.00096865, 0.        , 0.        ],
        [0.        , 0.00096865, 0.        ],
        [0.        , 0.        , 0.0019373 ]]),
 np.float64(0.0009686516666666668),
 np.float64(0.0009686516666666668),
 np.float64(0.0019373033333333336))

In [ ]:
params = {
    'm': 1.625,
    'Jx': Jx,
    'Jy': Jy,
    'Jz': Jz,
    'l': L,
    'g': 9.81,
}

Derive the equations of motion:

In [ ]:
# components of position (meters)
p_x, p_y, p_z = sym.symbols('p_x, p_y, p_z')

# yaw, pitch, roll angles (radians)
psi, theta, phi = sym.symbols('psi, theta, phi')

# components of linear velocity (meters / second)
v_x, v_y, v_z = sym.symbols('v_x, v_y, v_z')
v_in_body = sym.Matrix([v_x, v_y, v_z])

# components of angular velocity (radians / second)
w_x, w_y, w_z = sym.symbols('w_x, w_y, w_z')
w_in_body = sym.Matrix([w_x, w_y, w_z])

# components of net rotor torque
tau_x, tau_y, tau_z = sym.symbols('tau_x, tau_y, tau_z')

# net rotor force
f_z = sym.symbols('f_z')

# parameters
m = sym.nsimplify(params['m'])
Jx = sym.nsimplify(params['Jx'])
Jy = sym.nsimplify(params['Jy'])
Jz = sym.nsimplify(params['Jz'])
l = sym.nsimplify(params['l'])
g = sym.nsimplify(params['g'])
J = sym.diag(Jx, Jy, Jz)

# rotation matrices
Rz = sym.Matrix([[sym.cos(psi), -sym.sin(psi), 0], [sym.sin(psi), sym.cos(psi), 0], [0, 0, 1]])
Ry = sym.Matrix([[sym.cos(theta), 0, sym.sin(theta)], [0, 1, 0], [-sym.sin(theta), 0, sym.cos(theta)]])
Rx = sym.Matrix([[1, 0, 0], [0, sym.cos(phi), -sym.sin(phi)], [0, sym.sin(phi), sym.cos(phi)]])
R_body_in_world = Rz @ Ry @ Rx

# angular velocity to angular rates
ex = sym.Matrix([[1], [0], [0]])
ey = sym.Matrix([[0], [1], [0]])
ez = sym.Matrix([[0], [0], [1]])
M = sym.simplify(sym.Matrix.hstack((Ry @ Rx).T @ ez, Rx.T @ ey, ex).inv(), full=True)

# applied forces
f_in_body = R_body_in_world.T @ sym.Matrix([[0], [0], [-m * g]]) + sym.Matrix([[0], [0], [f_z]])

# applied torques
tau_in_body = sym.Matrix([[tau_x], [tau_y], [tau_z]])

# equations of motion
f = sym.Matrix.vstack(
    R_body_in_world @ v_in_body,
    M @ w_in_body,
    (1 / m) * (f_in_body - w_in_body.cross(m * v_in_body)),
    J.inv() @ (tau_in_body - w_in_body.cross(J @ w_in_body)),
)

f = sym.simplify(f, full=True)

The equations of motion have this form:

$$\begin{bmatrix} \dot{p}_x \\ \dot{p}_y \\ \dot{p}_z \\ \dot{\psi} \\ \dot{\theta} \\ \dot{\phi} \\ \dot{v}_x \\ \dot{v}_y \\ \dot{v}_z \\ \dot{w}_x \\ \dot{w}_y \\ \dot{w}_z \end{bmatrix} = f\left(p_x, p_y, p_z, \psi, \theta, \phi, v_x, v_y, v_z, w_x, w_y, w_z, \tau_x, \tau_y, \tau_z, f_z \right)$$

## Sensor model

Define the sensor model.

In [ ]:
# Position of drone in world frame
p_in_world = sym.Matrix([p_x, p_y, p_z])

# Position of markers in body frame
a_in_body = sym.Matrix([0, l, 0])  # <-- marker on left rotor
b_in_body = sym.Matrix([0, -l, 0]) # <-- marker on right rotor

# Position of markers in world frame
a_in_world = p_in_world + R_body_in_world @ a_in_body
b_in_world = p_in_world + R_body_in_world @ b_in_body

# Sensor model
g = sym.simplify(sym.Matrix.vstack(a_in_world, b_in_world))

The sensor model has this form:

$$o = g(p_x, p_y, p_z, \psi, \theta, \phi)$$


Define the system's symbolic state `x` and control input `u`, then compute the Jacobians of the nonlinear dynamics `f(x,u)` and output function `g(x)` with respect to `x` and `u`. These Jacobians are used to form the linearized system matrices `A`, `B`, and `C` around a nominal operating point.


In [ ]:
# Define the state vector `x` as a symbolic matrix containing position (p), orientation (Euler angles), linear velocity (v), and angular velocity (w)
x = Matrix([p_x, p_y, p_z, psi, theta, phi, v_x, v_y, v_z, w_x, w_y, w_z])

# Define the input vector `u` as a symbolic matrix containing control inputs: torques (tau) and thrust force (f_z)
u = Matrix([tau_x, tau_y, tau_z, f_z])

# Create a callable function `An` that returns the Jacobian of the dynamics `f` w.r.t. state `x`, given specific values for state and input
An = lambdify((p_x, p_y, p_z, psi, theta, phi, v_x, v_y, v_z, w_x, w_y, w_z, f_z, tau_x, tau_y, tau_z), f.jacobian(x))

# Create a callable function `Bn` that returns the Jacobian of the dynamics `f` w.r.t. input `u`, given specific values for state and input
Bn = lambdify((p_x, p_y, p_z, psi, theta, phi, v_x, v_y, v_z, w_x, w_y, w_z, f_z, tau_x, tau_y, tau_z), f.jacobian(u))

# Create a callable function `Cn` that returns the Jacobian of the output `g` w.r.t. state `x`, given values for the state
Cn = lambdify((p_x, p_y, p_z, psi, theta, phi), g.jacobian(x))

# Evaluate the Jacobian of the system dynamics `f` with respect to the state vector `x`, at the point (0,...,0, 0.5*9.81, 0, 0, 0)
A = An(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, .5*9.81, 0, 0, 0)

# Evaluate the Jacobian of the system dynamics `f` with respect to the control input vector `u`, at the same nominal point
B = Bn(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, .5*9.81, 0, 0, 0)

# Evaluate the Jacobian of the output function `g` with respect to the state vector `x`, at the point (0,...,0)
C = Cn(0, 0, 0, 0, 0, 0)


Check if the system is controllable and observable

In [ ]:
def w(A, B):
    W = B                                      # Start with the first column of the controllability/observability matrix
    n = A.shape[0]                             # Get the system order (number of states)
    for i in range(1, n):                      # Loop from 1 to n-1
        col = np.linalg.matrix_power(A, i) @ B # Compute A^i * B
        W = np.block([W, col])                 # Append the result as a new column block
    return W                                   # Return the full controllability/observability matrix
Wc = w(A, B)  # Compute the controllability matrix for (A, B)
Wo = w(A.T, C.T).T  # Compute the observability matrix for (A, C). Note: it's built as the transpose, so we transpose back
print(np.linalg.matrix_rank(Wc), Wc.shape, np.linalg.matrix_rank(Wo), Wo.shape)
# Print the rank and shape of the controllability and observability matrices to check if the system is controllable/observable
A.shape, B.shape, C.shape

The controllability matrix `Wc` has shape (12, 48) and rank 12, and the observability matrix `Wo` has shape (72, 12) and rank 12. Since both ranks equal the number of states (12), the system is fully controllable and observable.


Computee the state feedback gain `K` using the Linear Quadratic Regulator (LQR) method to stabilize the system, and the observer gain `L` using the dual LQR formulation. The weighting matrices `Q`, `R`, `Qo`, and `Ro` are selected to balance state regulation, control effort, and estimation performance.


In [ ]:
# Calculate state feedback gain K using the continuous-time LQR method
def lqr(A, B, Q, R):
    P = solve_continuous_are(A, B, Q, R)       # Solve the continuous-time Algebraic Riccati Equation (ARE)
    K = np.linalg.inv(R) @ B.T @ P             # Compute the optimal gain matrix K
    return K

# Design parameters for LQR controller
Q = np.diag([1,1,1,100,100,100,.1,.1,.1,1,1,1])*1   # State cost matrix (penalize orientation heavily)
R = np.diag([1000,1000,1000,.01])                             # Control cost matrix (penalize torques more than thrust)

# Design parameters for observer (dual LQR)
Qo = np.diag([1,1,1,1,1,1])*10              # Output noise covariance (larger = trust model more)
Ro = np.diag([1,1,1,1,1,1,1,1,1,1,1,1])*.01  # State noise covariance (smaller = trust measurements more)

K = lqr(A, B, Q, R)                                               # Compute LQR state feedback gain
L = lqr(A.T, C.T, np.linalg.inv(Ro), np.linalg.inv(Qo)).T        # Compute observer gain using dual LQR (Kalman-like)

# Optional: Check eigenvalues to confirm stability
# np.linalg.eigvals(A - B @ K), np.linalg.eigvals(A - L @ C)

## Environment setup

Import modules.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import secrets
import ae353_drone


For this report, a random seed was generated and saved. We saved the seed so the results obtained in the experiment are replicable.

In [ ]:
seed = 3215363112 #Seed with which simulator will be created

Create simulator with seed.

In [ ]:
simulator = ae353_drone.Simulator(seed=seed)

Add a camera view. This view will be displayed in its own browser window.

In [ ]:
simulator.add_view(
    'my_start_view',  # name of view (must be unique)
    'start',          # type of view (start, top, right, left, or back)
)

## Adding the drone

Clear all drones (there aren't any yet, if you are running this notebook from the start, but we call this function just in case).

In [ ]:
simulator.clear_drones()

Define the controller for the drone.

In [ ]:
class Controller:
    def __init__(self):
        # Variables to log during simulation (for plotting or analysis)
        self.variables_to_log = ['tar', 'y','xh']
        
        # System matrices for observer and controller
        self.A = A
        self.B = B
        self.C = C
        self.K = K                  # State feedback gain
        self.L = L                  # Observer gain
        self.P = A - self.L @ C     # Observer dynamics matrix A-LC
        self.ne = np.array([0,0,0,.5*9.81])  # Equilibrium input (e.g., gravity compensation thrust)

    def get_color(self):
        # RGB color for drone display (this one is red)
        return [1., 0., 0.]

    def reset(self, p_x, p_y, p_z, yaw):
        # Initialize estimated state x̂ and desired state x_d
        self.xh = np.array([p_x,p_y,p_z, yaw,0.,0., 0.,0.,0., 0.,0.,0.])  # estimated state
        self.xd = np.array([p_x,p_y,1., yaw,0.,0., 0.,0.,0., 0.,0.,0.])   # desired hover state at z=1
        self.close = False                                               # proximity flag for trajectory phases
        self.tar = np.array([0.,0.,0.])                                   # target position

    def settar(self, pr, dr, last, po):
        # Target update logic based on whether this is the last ring
        p1d, p2d = 0.75, 1.0    # position gains
        p1r, p2r = 0.5, 0.5     # radius thresholds

        if last:
            # Landing logic
            if not self.close:
                if np.linalg.norm(self.xh[0:2]) > 2.5:
                    self.tar = np.array([0.,0.,1.])
                else:
                    self.tar = np.array([0.,0.,0.])
            else:
                # Once landed, ascend again
                if np.linalg.norm(self.xh[0:3] - self.tar) < p2r:
                    self.tar = np.array([0.,0.,1.])
                    self.close = False
        else:
            # Ring navigation logic
            pdiff = pr - self.xh[0:3]
            nor = np.dot(pdiff, dr) * dr           # project onto direction
            tan = pdiff - nor                      # orthogonal (tangent) component

            bend = dr
            if (np.linalg.norm(tan) > 1e-3 and np.linalg.norm(nor) > 1e-3):
                ang = min(np.linalg.norm(tan)/np.linalg.norm(nor), 1)
                bend = dr + ang * tan / np.linalg.norm(tan)
                bend /= np.linalg.norm(bend)

            temp = pr - p1d * bend
            # Far from ring: go to pre-ring target
            if np.linalg.norm(self.xh[0:3] - (pr/3 + temp*2/3)) > p1r and not self.close:
                self.tar = temp
            elif not self.close:
                # Close enough: pass ring
                self.tar = pr + bend * p2d
                self.close = True
            else:
                # Passed: set next pre-ring target
                if np.linalg.norm(self.xh[0:3] - self.tar) < p2r:
                    self.tar = temp
                    self.close = False

        # Update desired trajectory (position-based)
        pdes = self.incr(po)
        self.xd = np.block([pdes[0], pdes[1], pdes[2], 0.,0.,0., 0.,0.,0., 0.,0.,0.])

    def incr(self, po):
        # Compute next desired position incrementally based on current target
        pdiff = self.tar - self.xh[0:3]
        gain = 3. * np.log10(np.linalg.norm(pdiff)+6)  # speed gain
        return self.setsafe(po) + gain * pdiff / np.linalg.norm(pdiff)  # directional movement toward target

    def setsafe(self, po):
        # Compute collision-avoiding desired position using repulsion
        phat = self.xh[0:3]
        k_rep = 0.25  # repulsion strength
        k_des = 0.25  # movement strength
        r_drone = 0.25  # drone safety radius

        grad_h_rep = np.zeros(3)  # initialize repulsion gradient

        for q in po:  # loop through other drones
            pobst = q + r_drone * (phat - q) / np.linalg.norm(phat - q)  # buffer zone
            dgrad = (phat - pobst) / np.linalg.norm(phat - pobst)
            d = np.linalg.norm(phat - pobst) - r_drone
            grad_h_rep += -k_rep / d**2 * dgrad  # add repulsive gradient

        return phat - k_des * grad_h_rep  # updated safe position

    def run(self, pos_markers, pos_ring, dir_ring, is_last_ring, pos_others):
        # Main controller update function called at each timestep
        self.settar(pos_ring, dir_ring, is_last_ring, pos_others)  # update trajectory
        
        # Sensor measurement: difference between marker positions
        y = pos_markers - np.array([0., 7/40, 0., 0., -7/40, 0.])
        self.y = y

        # Compute control input using state feedback
        u = -self.K @ (self.xh - self.xd)
        
        # Add equilibrium input (e.g., gravity compensation)
        n = u + self.ne

        # Observer update: estimate state using observer dynamics
        self.xh += 0.04 * (self.P @ self.xh + self.B @ u + self.L @ y)

        # Return control inputs
        return n[0], n[1], n[2], n[3]


Practice run with result to make sure everything is working. This run has no display to make it quicker.

In [ ]:
simulator.reset()  
# Reset the simulator environment to its initial state (clears any prior simulation state)

simulator.clear_drones()  
# Remove all previously added drones from the simulator

simulator.disable_views()  
# Disable visual rendering (useful for speeding up headless or data-only runs)

for i in range(1):
    simulator.add_drone(Controller, f'drone_1', 'template.png')
    # Add one drone to the simulation using the custom Controller class
    # Assign it a unique name (e.g., 'drone_1') and appearance template
    # For function is used now "Trivially" but will come in useful in a later cell

simulator.reset()
# Reset again after drone(s) have been added to ensure a clean simulation start

simulator.run(
    max_time=150.,       # Run the simulation for 150 seconds maximum
    print_debug=True     # Enable console output for debugging/logging
)

data = simulator.get_data('drone_1')  
# Retrieve simulation data for drone_1 (state history, control inputs, etc.)


Optional: Real time run to see drone go around the racetrack

In [ ]:
#simulator.reset()
#simulator.clear_drones()

#for i in range(1):
    #simulator.add_drone(Controller, f'drone_{i+1}', 'template.png')

#simulator.reset()
#simulator.run(
    #max_time=150.,       # <-- if None, then simulation will run until all drones fail or finish
    #print_debug=True,  # <-- if False, then nothing will be printed (good for data collection)
#)

To analyze more complex data, we must start retrieving data.

In [ ]:
(
    did_it_fail,
    did_it_finish,
    what_time_did_it_finish,
) = simulator.get_result('drone_1')

Test the drone by itself over 100 tries to get success rate.

In [ ]:
simulator.clear_drones()     # Remove any previously added drones to start fresh
simulator.disable_views()    # Disable visual output for faster, headless simulation runs
results = []  # This list will store data for each trial

simulator.add_drone(Controller, 'drone_1', 'template.png')
# Add one drone to the simulation using the specified controller and appearance

for i in range(100):
    simulator.reset()  # Reset the simulator environment for a clean trial
    simulator.run(max_time=150., print_debug=False)  # Run simulation up to 150s (or until drone fails)

    # Get the result from this trial (returns 3 values)
    did_it_fail, did_it_finish, what_time_did_it_finish = simulator.get_result('drone_1')

    # Store result in the results list
    results.append({
        'trial': i + 1,
        'failed': did_it_fail,
        'finished': did_it_finish,
        'final_time': what_time_did_it_finish
    })

# === Compute Average Finish Time ===
all_times = [r['final_time'] for r in results if r['final_time'] is not None]
avg_time = sum(all_times) / len(all_times) if all_times else float('nan')

# === Compute Success Rate ===
fail_count = sum(r['failed'] for r in results)
success_rate = 100 * (len(results) - fail_count) / len(results)

# === Display Results ===
print(f"Average Finish Time over {len(all_times)} trials: {avg_time:.2f} seconds")
print(f"Success Rate: {success_rate:.2f}%")



Plot histogram of controller run times.It will take the data from a previously ran trial.

In [ ]:
plt.hist(data['run_time'])
plt.ticklabel_format(style='scientific', scilimits=(0, 0), axis='x')
plt.tick_params(labelsize=14)
plt.xlabel('Run time (s)', fontsize=14)
plt.ylabel('Count', fontsize=14)
plt.tight_layout()
plt.show()

Plot the error in the state. This too will take data from a previously ran trial.

In [ ]:
# Assume data['xh'], data['xd'], data['xhat'] are already available
xh = np.array(data['xh'])   # true state
xd = np.array(data['tar']) # desired state

# Compute errors
state_error = xh[:, :3] - xd

# Time vector (assuming dt = 0.04 s)
t = np.arange(len(xh)) * 0.04

# Custom labels and linestyles
labels = ['x error', 'y error', 'z error']
linestyles = ['-', '--', ':']  # solid, dashed, dotted

# Plot state error for first 3 states only
plt.figure(figsize=(10, 6))
for i in range(3):
    plt.plot(t, state_error[:, i], label=labels[i], linestyle=linestyles[i])

plt.xlabel('Time (s)', fontsize=14)
plt.ylabel('Position Error (m)', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.legend(fontsize=13)
plt.grid()
plt.show()


Plot true position versus target position.This too will take data from a previously ran trial.

In [ ]:
# Get logs
xh = np.array(data['xh'])    # true state (N,12)
tar = np.array(data['tar'])   # target position (N,3)

# Extract x and y positions
xh_x = xh[:, 0]   # true x
xh_y = xh[:, 1]   # true y

tar_x = tar[:, 0]  # target x
tar_y = tar[:, 1]  # target y

# Plot
plt.figure(figsize=(8, 6))

# True position
plt.plot(xh_x, xh_y, label='True Position (xh)', linewidth=2)

# Target position
plt.plot(tar_x, tar_y, ':', label='Target Position (tar)', linewidth=2)

# Font settings
plt.xlabel('X position (m)', fontsize=14)
plt.ylabel('Y position (m)', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Legend on the left
plt.legend(fontsize=13, bbox_to_anchor=(0.4, 1))

plt.grid(True)
plt.axis('equal')  # Keep aspect ratio square
plt.tight_layout()
plt.show()

This section runs 100 trials of the drone simulation to evaluate the performance of the state observer. For each trial, it computes the root-mean-square (RMS) error between the measured and estimated states. The results are visualized using grouped histograms to show the distribution of errors across position and orientation states.


In [ ]:
# === Simulator setup ===
simulator.reset()                               # Reset the simulator to a clean state
simulator.clear_drones()                        # Remove any previously added drones
simulator.disable_views()                       # Disable visualization for faster execution

# === Trial setup ===
num_trials = 100                                # Number of simulation trials to run
all_rms_errors = []                             # List to store RMS errors for each trial

# === Main simulation loop ===
for trial in range(num_trials):
    simulator.reset()                           # Reset environment for this trial
    simulator.clear_drones()                    # Ensure no leftover drones
    simulator.disable_views()                   # Keep it headless

    simulator.add_drone(Controller, 'drone_1', 'template.png')  # Add a single drone

    simulator.reset()                           # Final reset after drone addition
    simulator.run(max_time=150., print_debug=False)  # Run the simulation for max 150 seconds

    data = simulator.get_data('drone_1')        # Get logged data from the drone

    try:
        xh = np.array(data['xh'])[:, :6]        # Extract estimated state (N x 6)
        y = np.array(data['y'])                 # Extract measurements (should also be N x 6)

        observer_error = y - xh                 # Element-wise error between measurement and estimate
        rms_error = np.sqrt(np.mean(observer_error**2, axis=0))  # RMS error across time per state
        all_rms_errors.append(rms_error)        # Store the result

    except Exception as e:
        print(f"Trial {trial+1} failed: {e}")    # If something goes wrong, skip this trial
        continue

# === Organize the data ===
all_rms_errors = np.array(all_rms_errors)       # Convert list to array (shape: num_trials x 6)

# === Define plot colors ===
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown']  # For bar groups

# === Grouped histogram plotting function ===
def plot_grouped_histogram(data, indices, title, legend_labels=None):
    num_states = len(indices)                                           # Number of states to plot
    max_val = np.max(data[:, indices])                                  # Max value across selected states
    bins = np.linspace(0, max_val, 16)                                  # Bin edges (15 bins)
    bin_centers = (bins[:-1] + bins[1:]) / 2                            # Center of each bin
    bin_width = bins[1] - bins[0]                                       # Width of each bin
    bar_width = bin_width / (num_states + 1)                            # Bar width for grouping

    fig, ax = plt.subplots(figsize=(10, 6))                             # Set figure size

    for i, idx in enumerate(indices):
        counts, _ = np.histogram(data[:, idx], bins=bins)              # Compute histogram counts
        shift = (i - num_states / 2) * bar_width + bar_width / 2       # Shift each bar group horizontally
        label = legend_labels[i] if legend_labels else f'$x_{idx}$'    # Label for legend
        ax.bar(bin_centers + shift, counts, width=bar_width,
               color=colors[idx], label=label, edgecolor='black')      # Plot each group

    ax.set_title(title, fontsize=16)                                   # Title of the plot
    ax.set_xlabel('RMS Error', fontsize=14)                            # X-axis label
    ax.set_ylabel('Frequency', fontsize=14)                            # Y-axis label
    ax.legend(loc='center left', bbox_to_anchor=(-0, 0.9))             # Legend placement
    ax.grid(True)                                                      # Add grid lines
    plt.tight_layout()                                                 # Clean layout
    plt.show()                                                         # Display plot

# === Plot for position states: x, y, z ===
plot_grouped_histogram(
    all_rms_errors, 
    indices=[0, 1, 2], 
    title='RMS Error Distribution for Position States',
    legend_labels=[r'$x$', r'$y$', r'$z$']
)

# === Plot for orientation states: ψ, θ, φ ===
plot_grouped_histogram(
    all_rms_errors, 
    indices=[3, 4, 5], 
    title='RMS Error Distribution for Orientation States',
    legend_labels=[r'$\psi$', r'$\theta$', r'$\phi$']
)
